# Chapter 16 — Bayesian vs Frequentist Thinking
*Track 3: Data Scientists*

## 🎯 Learning Objectives
- Understand the philosophical divide between Bayesian and Frequentist statistics
- Compute Bayesian credible intervals and frequentist confidence intervals
- See how they diverge and when each is appropriate

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.special import beta as beta_fn

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review

| | Frequentist | Bayesian |
|--|-------------|---------|
| Parameters | Fixed, unknown constants | Random variables with distributions |
| Data | Random (repeated sampling) | Fixed (observed) |
| Inference | Confidence intervals, p-values | Posterior distribution, credible intervals |
| Prior info | Not used | Explicitly incorporated |
| Statement | "95% of such intervals contain θ" | "P(θ in CI | data) = 0.95" |

In [ ]:
# Frequentist vs Bayesian: coin flip example
n_flips = 20; n_heads = 14

# Frequentist CI (Wilson interval)
p_hat = n_heads / n_flips
z = stats.norm.ppf(0.975)
ci_lo = (p_hat + z**2/(2*n_flips) - z*np.sqrt(p_hat*(1-p_hat)/n_flips + z**2/(4*n_flips**2))) /         (1 + z**2/n_flips)
ci_hi = (p_hat + z**2/(2*n_flips) + z*np.sqrt(p_hat*(1-p_hat)/n_flips + z**2/(4*n_flips**2))) /         (1 + z**2/n_flips)
print(f"Frequentist 95% CI: [{ci_lo:.3f}, {ci_hi:.3f}]")

# Bayesian: Beta(1,1) prior → Beta(1+14, 1+6) posterior
a_prior, b_prior = 1, 1
a_post = a_prior + n_heads
b_post = b_prior + (n_flips - n_heads)
cred_lo, cred_hi = stats.beta.ppf([0.025, 0.975], a_post, b_post)
print(f"Bayesian 95% credible interval: [{cred_lo:.3f}, {cred_hi:.3f}]")

p_range = np.linspace(0, 1, 300)
plt.figure(figsize=(9, 4))
plt.plot(p_range, stats.beta.pdf(p_range, a_post, b_post), "b-", lw=2, label="Posterior")
plt.axvspan(cred_lo, cred_hi, alpha=0.2, color="blue", label="95% Credible Interval")
plt.axvline(p_hat, color="red", linestyle="--", label=f"Freq. MLE p̂={p_hat:.2f}")
plt.xlabel("p"); plt.ylabel("Density"); plt.title("Bayesian vs Frequentist — Coin Flip")
plt.legend(); plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Bayes' Theorem in Practice

$$p(\theta | \mathcal D) \propto p(\mathcal D | \theta)\, p(\theta)$$

For $n$ Bernoulli trials with $k$ successes and $\text{Beta}(\alpha, \beta)$ prior:
$$p(p | k, n) = \text{Beta}(\alpha + k,\, \beta + n - k)$$

This is a **conjugate prior** — prior and posterior have the same functional form.

In [ ]:
# Show posterior updating step by step
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
observations = [0, 5, 14, 50]
cumulative_heads = [0, 3, 8, 35]
prior_a, prior_b = 1, 1
for ax, (n_obs, k) in zip(axes, zip(observations, cumulative_heads)):
    a_p = prior_a + k
    b_p = prior_b + (n_obs - k)
    p_vals = np.linspace(0, 1, 300)
    ax.plot(p_vals, stats.beta.pdf(p_vals, a_p, b_p), lw=2)
    ax.axvline(a_p/(a_p+b_p), color="red", linestyle="--", label=f"Mean={a_p/(a_p+b_p):.2f}")
    ax.set_title(f"After {n_obs} flips
({k} heads)")
    ax.legend(fontsize=8)
plt.suptitle("Bayesian Updating — Posterior Evolves with Data", fontweight="bold")
plt.tight_layout(); plt.show()

## 3-6. Simulation, Visualisation, Real Dataset & Practice

In [ ]:
# Misinterpretation demo: CI doesn't mean what most people think
n_simulations = 1000; n_data = 30; true_mu = 5.0
covered = 0
plt.figure(figsize=(12, 4))
for i in range(100):
    data = rng.normal(true_mu, 1, n_data)
    se = data.std(ddof=1) / np.sqrt(n_data)
    lo, hi = data.mean() - 1.96*se, data.mean() + 1.96*se
    color = "blue" if lo <= true_mu <= hi else "red"
    plt.plot([i, i], [lo, hi], color=color, alpha=0.6, lw=0.8)
for i in range(1000):
    data = rng.normal(true_mu, 1, n_data)
    se = data.std(ddof=1) / np.sqrt(n_data)
    lo, hi = data.mean() - 1.96*se, data.mean() + 1.96*se
    covered += (lo <= true_mu <= hi)
plt.axhline(true_mu, color="black", linewidth=1.5, label=f"True μ={true_mu}")
plt.title(f"95% CI Coverage: {covered/1000:.1%} of 1000 intervals contain μ")
plt.xlabel("Simulation"); plt.ylabel("Interval")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Practice problems
# P1: Compute posterior mean and MAP for Beta(3,7) prior, 20 flips, 12 heads
a0, b0 = 3, 7; k, n = 12, 20
a1, b1 = a0+k, b0+(n-k)
post_mean = a1 / (a1+b1)
post_map  = (a1-1)/(a1+b1-2) if a1+b1>2 else 0.5
print(f"P1 posterior: Beta({a1},{b1})")
print(f"   Mean={post_mean:.4f}, MAP={post_map:.4f}")

# P2: Compare credible vs confidence interval width as n grows
for n_obs in [10, 50, 200, 1000]:
    k_obs = int(n_obs * 0.6)
    # Bayesian (uniform prior)
    a_p, b_p = 1+k_obs, 1+(n_obs-k_obs)
    cred = stats.beta.ppf(0.975, a_p, b_p) - stats.beta.ppf(0.025, a_p, b_p)
    # Frequentist
    ph = k_obs/n_obs
    conf = 2 * 1.96 * np.sqrt(ph*(1-ph)/n_obs)
    print(f"n={n_obs:>5}: Credible width={cred:.4f}, CI width={conf:.4f}")